# Accuracy verification: best-of-n on AIME (Llama-3.2-1B-Instruct)

Reads the best-of-n completion JSONL files for **vllm** (baseline) and **ours**
from
`data/meta-llama/Llama-3.2-1B-Instruct/aimo-validation-aime/accuracy_verify/`,
computes accuracy for each beam budget `n`, and reports it as a table and a
grouped bar chart.

- Model: `meta-llama/Llama-3.2-1B-Instruct`
- Dataset: `aimo-validation-aime` (90 problems)
- Approach: best-of-n

Grading uses the repo's `sal/utils/math.py`: `memoized_canonical_form`
normalizes both the prediction and the gold `answer` to a sympy canonical form,
and the two are compared for symbolic equivalence (`extract_answer` first pulls
the boxed answer out of a completion).

**Correctness criterion:** kernel optimizations change floating-point results,
so `ours` will not be row-by-row identical to `vllm`. AIME is only 90 problems
and very hard for a 1B model — expect low accuracy and noise-dominated gaps.

In [ ]:
import json
import re
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Repo layout: this notebook lives in <repo>/scripts/, package is <repo>/src/sal
REPO = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

# Grading primitives from the repo (sal/utils/math.py + the answer parser).
from sal.utils.qwen_math_parser import extract_answer
from sal.utils.math import memoized_canonical_form

# MODEL = "meta-llama/Llama-3.2-1B-Instruct"
MODEL = "Qwen/QWen2.5-1.5B-Instruct"
# DATASET = "aimo-validation-aime"
DATASET = "MATH-500"
METHODS = ["vllm", "ours"]

# All best_of_n_completions_{vllm,ours}_n*.jsonl files live in this folder.
BON_DIR = REPO / "data" / MODEL / DATASET / "accuracy_verify"

FIG_DIR = REPO / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("repo    :", REPO)
print("bon dir :", BON_DIR, "(exists)" if BON_DIR.is_dir() else "(MISSING)")

In [ ]:
# --- grading helpers ---------------------------------------------------

def canon(s):
    """Canonical (symbolically-normalized) form, with a safe fallback."""
    try:
        return memoized_canonical_form(str(s))
    except Exception:
        return str(s).strip()


def is_correct(pred_text, gold_canon):
    """True if the answer extracted from `pred_text` matches the gold answer.

    `pred_text` may be a full completion or an already-boxed string;
    extract_answer handles both.
    """
    if pred_text is None or pred_text == "":
        return False
    pred_ans = extract_answer(str(pred_text), "math")
    if pred_ans is None or pred_ans == "":
        return False
    return canon(pred_ans) == gold_canon


def parse_name(path):
    """best_of_n_completions_<method>_n<N>.jsonl -> (method, N) or None."""
    m = re.match(r"best_of_n_completions_(\w+?)_n(\d+)\.jsonl$", path.name)
    return (m.group(1), int(m.group(2))) if m else None


def prm_was_used(data):
    """A run is PRM-scored if any aggregated score is non-zero. PRM-disabled
    runs write placeholder scores of 0.0 for every beam."""
    return any(v for d in data for v in d.get("agg_scores", []))

In [ ]:
# --- 1. discover the vllm + ours json files ----------------------------
files = []
if BON_DIR.is_dir():
    for p in sorted(BON_DIR.glob("best_of_n_completions_*_n*.jsonl")):
        parsed = parse_name(p)
        if parsed and parsed[0] in METHODS:
            files.append((parsed[0], parsed[1], p))

files.sort(key=lambda t: (t[0], t[1]))
for method, n, p in files:
    print(f"  {method:5s}  n={n:<4d}  {p.name}")
print(f"\n{len(files)} files to evaluate")
if not files:
    print(f"\n[!] No best-of-n files in {BON_DIR}")

In [ ]:
# --- 2. compute accuracy for each beam setting -------------------------
# Three predictions, all already present in the JSONL:
#   pred            -> best-of-n: PRM-argmax winner over all n completions
#   pred_weighted@n -> PRM score-weighted vote
#   pred_maj@n      -> plain majority vote (does NOT need the PRM)
rows = []
for method, n, path in files:
    with open(path) as fh:
        data = [json.loads(line) for line in fh]

    gold = [canon(d["answer"]) for d in data]
    n_problems = len(data)
    prm_ok = prm_was_used(data)

    bon = sum(is_correct(d["pred"], g) for d, g in zip(data, gold))

    wcol, mcol = f"pred_weighted@{n}", f"pred_maj@{n}"
    weighted = (sum(is_correct(d.get(wcol), g) for d, g in zip(data, gold))
                if wcol in data[0] else None)
    majority = (sum(is_correct(d.get(mcol), g) for d, g in zip(data, gold))
                if mcol in data[0] else None)

    rows.append({
        "method": method,
        "n": n,
        "problems": n_problems,
        "prm_scored": prm_ok,
        "best_of_n": bon / n_problems,
        "weighted": weighted / n_problems if weighted is not None else None,
        "majority": majority / n_problems if majority is not None else None,
    })
    flag = "" if prm_ok else "  [PRM DISABLED -> best_of_n/weighted invalid]"
    print(f"{method:5s} n={n:<4d}  best-of-n={bon}/{n_problems}  "
          f"majority={majority}/{n_problems}{flag}")

df = pd.DataFrame(rows).sort_values(["method", "n"]).reset_index(drop=True)
df

In [ ]:
# --- 3a. report: vllm vs ours table ------------------------------------
if not df.empty:
    # best-of-n is only trustworthy when the PRM ran; otherwise fall back to
    # majority vote as the headline metric.
    METRIC = "best_of_n" if df["prm_scored"].all() else "majority"
    if METRIC != "best_of_n":
        print("[!] Some files are PRM-disabled (placeholder scores).")
        print("    Reporting MAJORITY vote instead of best-of-n.\n")

    pivot = df.pivot(index="n", columns="method", values=METRIC)
    pivot = pivot.reindex(columns=[m for m in METHODS if m in pivot.columns])
    if {"vllm", "ours"}.issubset(pivot.columns):
        pivot["delta (ours-vllm)"] = pivot["ours"] - pivot["vllm"]

    print(f"AIME accuracy ({METRIC}, %)  -  Llama-3.2-1B-Instruct\n")
    print((pivot * 100).round(2).to_string())
else:
    METRIC = "best_of_n"
    print("No data to report.")

In [ ]:
# --- 3b. report: grouped bar chart -------------------------------------
# ================= FIGURE STYLE CONTROLS (tweak freely) =================
FIG_SIZE        = (8, 3.5)     # figure size in inches (width, height)
FONTSIZE_LABEL  = 14         # x / y axis label font size
FONTSIZE_LEGEND = 14         # legend font size
FONTSIZE_TICK   = 14         # axis tick (values) font size
BAR_WIDTH       = 0.30       # width of each individual bar
INTRA_GAP       = 0.04       # gap between the two bars within one group
INTER_GAP       = 0.40       # gap between adjacent beam-count groups
Y_LIM           = (0, 80)   # y-axis range (min, max) in %; set None for auto
# ========================================================================

# per-method bar color and legend label
colors = {"vllm": "#7fc4f5", "ours": "#fac14f"}
LEGEND_LABELS = {"vllm": "HuggingFace", "ours": "AccTTS"}

if not df.empty:
    beams = sorted(df["n"].unique())
    # group stride = total span of one group (2 bars + intra gap) + inter gap
    group_w = 2 * BAR_WIDTH + INTRA_GAP
    stride = group_w + INTER_GAP
    centers = [i * stride for i in range(len(beams))]
    off = (BAR_WIDTH + INTRA_GAP) / 2   # each bar's offset from group center

    fig, ax = plt.subplots(figsize=FIG_SIZE)
    for sign, method in zip((-1, +1), METHODS):
        sub = df[df["method"] == method].set_index("n")
        vals = [sub.loc[b, METRIC] * 100 if b in sub.index else 0.0
                for b in beams]
        xs = [c + sign * off for c in centers]
        ax.bar(xs, vals, width=BAR_WIDTH, label=LEGEND_LABELS.get(method, method),
               color=colors.get(method))

    ax.set_xticks(centers)
    ax.grid(True, axis='y', which='both', linestyle='--', alpha=0.6)
    ax.set_xticklabels([str(b) for b in beams], fontsize=FONTSIZE_TICK)
    ax.tick_params(axis="y", labelsize=FONTSIZE_TICK)
    ax.set_xlabel("#Beams", fontsize=FONTSIZE_LABEL)
    ax.set_ylabel("MATH-500 Accuracy (%)", fontsize=FONTSIZE_LABEL)
    if Y_LIM is not None:
        ax.set_ylim(*Y_LIM)
    ax.legend(fontsize=FONTSIZE_LEGEND)
    fig.tight_layout()

    out_png = FIG_DIR / "accuracy_verify.png"
    fig.savefig(out_png, dpi=300)
    print("saved:", out_png)
    plt.show()
else:
    print("No data to plot.")